# H2 · A charge near a dielectric: where the local Poisson equation beats $1/r$
**Series H, Post 2** &nbsp;|&nbsp; tutorial notebook for the post of the same name

**The question.** Two electrons repel with the Coulomb interaction $1/r$, and Post H1 showed that a *local* equation on a grid (Poisson's equation) reproduces that interaction. But for two charges in empty space, plain $1/r$ is just as good, so what is the local equation *for*? This notebook builds the smallest example in which plain $1/r$ **cannot** give the right answer and the local equation can: **one charge near a dielectric** (a polarizable material). The dielectric responds to the charge, attracts it, and the energy of that attraction is invisible to any pair sum of $1/r$ terms. We solve the Poisson equation with a *position-dependent* dielectric constant $\epsilon(\mathbf r)$ by **conjugate gradients** and compare with the exact image-charge solution.

> **You need** high-school calculus and the idea of an electric potential. Dielectrics, image charges, the variable-coefficient stencil, conjugate gradients and the convergence order are all explained when they first appear. (Post H1 explains Poisson's equation and the grid stencil; a short recap is given below.)
>
> **Two parts.** **Part I** needs NumPy, SciPy and matplotlib and takes about a minute on a CPU. **Part II** is optional: independent checks with SciPy.
>
> **How to use it.** Each section starts with a short **Stop and predict** question. Answer it on paper before you run the cell.

| Section | What you do |
|---|---|
| 1 | what $1/r$ misses |
| 2 | the exact answer: image charges (and a check of the interface conditions) |
| 3 | the equation on a grid with a position-dependent $\epsilon$; why the *harmonic* mean |
| 4 | build the matrix and solve it by conjugate gradients |
| 5 | the reaction potential and its convergence |
| 6 | the dependence on $\epsilon$ and on the distance |
| 7 | why the fast sine transform cannot be used here |

**Recap of Post H1 (two paragraphs).** The potential $\phi$ of a charge density $\rho$ obeys **Poisson's equation** $-\nabla^2\phi=4\pi\rho$ (atomic units). On a grid of spacing $h$ the Laplacian becomes the **7-point stencil**: the equation at a grid point involves only the point and its six neighbours, so the system of equations is a *sparse* matrix equation $A\phi=b$ with seven nonzero entries per row. **Conjugate gradients** (CG) solves such a system by repeatedly multiplying the matrix with a vector (a neighbour-to-neighbour operation), improving the guess at each step with the recurrence
$\alpha_k=\frac{r_k^Tr_k}{p_k^TAp_k},\ \phi_{k+1}=\phi_k+\alpha_kp_k,\ r_{k+1}=r_k-\alpha_kAp_k,\ \beta_k=\frac{r_{k+1}^Tr_{k+1}}{r_k^Tr_k},\ p_{k+1}=r_{k+1}+\beta_kp_k$;
the residual $r=b-A\phi$ measures how much each equation is still violated. It needs only that $A$ be symmetric and positive definite.

## Helper functions
One cell with the numerics (the variable-coefficient stencil matrix, the conjugate-gradient solver written out, the exact image-charge solution) and one with the figure code.

In [ ]:
"""A point charge near a dielectric half-space: variable-coefficient Poisson equation on a grid, solved by conjugate gradients."""
import math, time, numpy as np, scipy.sparse as sp, scipy.sparse.linalg as spla
def image_charges(eps2):
    """vacuum above (z > 0), dielectric eps2 below (z < 0); unit charge at height d: image charge and transmitted charge"""
    return -(eps2 - 1) / (eps2 + 1), 2 / (1 + eps2)
def exact_potential(X, Y, Z, eps2, d, q=1.0):
    qi, qt = image_charges(eps2); r1 = np.sqrt(X ** 2 + Y ** 2 + (Z - d) ** 2); r2 = np.sqrt(X ** 2 + Y ** 2 + (Z + d) ** 2)
    with np.errstate(divide="ignore", invalid="ignore"): return np.where(Z > 0, q / r1 + q * qi / r2, q * qt / r1)
def grid_coordinates(h, d, L):
    """x, y nodes on a centred lattice; z nodes at (k + 1/2) h so that the interface z = 0 lies midway between two node planes"""
    nz = int(L / h) + 1; nxy = 2 * int(L / h) + 1; z = (np.arange(-nz, nz) + 0.5) * h; x = (np.arange(nxy) - (nxy - 1) / 2) * h; return x, x, z
def face_coefficient(a, b): return 2 * a * b / (a + b)          # harmonic mean of the two cells' epsilon: the flux through a face is like two conductances in series
def build(h, eps2, d, L=None):
    """interior unknowns, matrix A, right-hand side b of  sum_faces eps_f (phi_i - phi_j)/h^2 = 4 pi rho_i  with exact potential on the box walls"""
    L = L or max(3.0, 3 * d); x, y, z = grid_coordinates(h, d, L); X, Y, Z = np.meshgrid(x, y, z, indexing="ij"); shape = X.shape; n = X.size
    eps = np.where(Z < 0, eps2, 1.0); k0 = int(np.argmin(np.abs(z - d))); assert abs(z[k0] - d) < 1e-9, "choose h so that d/h is half-integer"; i0 = len(x) // 2
    rho = np.zeros(shape); rho[i0, i0, k0] = 1 / h ** 3; rhs = 4 * np.pi * rho; inner = np.zeros(shape, bool); inner[1:-1, 1:-1, 1:-1] = True; ex = exact_potential(X, Y, Z, eps2, d)
    idx = np.arange(n).reshape(shape); diag = np.zeros(shape); rows, cols, vals = [], [], []
    for ax in range(3):
        for sgn in (1, -1):
            sa = [slice(None)] * 3; sb = [slice(None)] * 3
            if sgn == 1: sa[ax] = slice(0, -1); sb[ax] = slice(1, None)
            else: sa[ax] = slice(1, None); sb[ax] = slice(0, -1)
            a, b = tuple(sa), tuple(sb); c = face_coefficient(eps[a], eps[b]) / h ** 2; diag[a] += c; both = inner[a] & inner[b]; rows.append(idx[a][both]); cols.append(idx[b][both]); vals.append(-c[both])
            wall = inner[a] & ~inner[b]; tmp = np.zeros(shape); tmp[a] = np.where(wall, c * ex[b], 0.0); rhs += tmp
    mask = inner.ravel(); m = -np.ones(n, int); m[mask] = np.arange(mask.sum())
    A = sp.coo_matrix((np.concatenate(vals), (m[np.concatenate(rows)], m[np.concatenate(cols)])), shape=(mask.sum(), mask.sum())).tocsr() + sp.diags(diag.ravel()[mask])
    return dict(A=A, b=rhs.ravel()[mask], mask=mask, shape=shape, ex=ex, pos=(i0, i0, k0), x=x, z=z, h=h, eps=eps, X=X, Y=Y, Z=Z)
def cg(A, b, tol=1e-11, history=False):
    """conjugate gradients written out (the recurrence of the post); returns the solution, the iteration count and the residual history"""
    x = np.zeros_like(b); r = b - A @ x; p = r.copy(); rs = r @ r; nb = np.linalg.norm(b); hist = [np.sqrt(rs) / nb]; it = 0
    while hist[-1] > tol and it < 20000:
        Ap = A @ p; al = rs / (p @ Ap); x += al * p; r -= al * Ap; rn = r @ r; p = r + (rn / rs) * p; rs = rn; it += 1; hist.append(np.sqrt(rs) / nb)
    return x, it, np.array(hist)
def potential(sys_, tol=1e-11):
    phi, it, hist = cg(sys_["A"], sys_["b"], tol); full = np.zeros(sys_["mask"].size); full[sys_["mask"]] = phi; full = full.reshape(sys_["shape"]); wall = ~sys_["mask"].reshape(sys_["shape"]); full[wall] = sys_["ex"][wall]; return full, it, hist
def reaction(h, eps2, d, tol=1e-11, L=None):
    """reaction potential at the charge = (grid with dielectric) - (grid in vacuum): the lattice self-value cancels"""
    s2 = build(h, eps2, d, L); s1 = build(h, 1.0, d, L); p2, it2, h2 = potential(s2, tol); p1, it1, _ = potential(s1, tol)
    return p2[s2["pos"]] - p1[s1["pos"]], it2, s2["A"].shape[0], p2, s2, h2

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
C = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
def fig_potential(D, d=1.0, eps2=4.0):
    h = d / 4.5; s2 = D.build(h, eps2, d); s1 = D.build(h, 1.0, d); p2, _, _ = D.potential(s2); p1, _, _ = D.potential(s1); i0 = s2["pos"][0]; x, z = s2["x"], s2["z"]
    fig, ax = plt.subplots(1, 2, figsize=(10.8, 4.0), gridspec_kw=dict(width_ratios=[1.15, 1])); diff = (p2 - p1)[:, i0, :]; sel = (np.abs(x) <= 2.4)[:, None] & (np.abs(z) <= 2.4)[None, :]
    im = ax[0].pcolormesh(x, z, diff.T, cmap="RdBu_r", shading="auto", vmin=-0.6, vmax=0.6); ax[0].axhline(0, color="k", lw=1.5); ax[0].plot([0], [d], "o", color="k"); ax[0].text(0.12, d, "charge", fontsize=8); ax[0].text(-2.3, -0.4, f"dielectric (eps = {eps2:g})", fontsize=8); ax[0].text(-2.3, 0.15, "vacuum", fontsize=8)
    ax[0].set(xlim=(-2.4, 2.4), ylim=(-2.4, 2.4), xlabel="x", ylabel="z", title="(a) what the dielectric adds to the potential\n(grid with dielectric minus grid in vacuum)", aspect="equal"); plt.colorbar(im, ax=ax[0], fraction=0.045)
    k = i0; zz = z; ex = D.exact_potential(0 * zz, 0 * zz, zz, eps2, d); ax[1].plot(zz, p2[k, k, :], "-", color=C["blue"], lw=2, label="grid, dielectric"); ax[1].plot(zz, p1[k, k, :], "-", color=C["gray"], lw=1.2, label="grid, vacuum (what 1/r gives)"); ax[1].plot(zz[::3], ex[::3], "o", color=C["red"], ms=3, label="exact (image charges)")
    ax[1].axvline(0, color="k", lw=1); ax[1].set(xlim=(-2.4, 2.4), ylim=(0, 3.2), xlabel="height z along the axis through the charge", ylabel="potential", title="(b) the potential along the axis"); ax[1].legend(frameon=False, fontsize=8, loc="upper left"); plt.tight_layout(); return fig
def fig_conv(R):
    rows = R["conv"]; fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.6)); cols = {2.0: C["green"], 4.0: C["blue"], 10.0: C["orange"]}
    for e in (2.0, 4.0, 10.0):
        r = [x for x in rows if x[0] == e]; hs = np.array([x[2] for x in r]); er = np.abs([x[5] - x[6] for x in r]); ax[0].loglog(hs, er, "o-", color=cols[e], label=f"eps = {e:g}")
    ax[0].loglog(hs, er[-1] * (hs / hs[-1]) ** 2 * 1.3, "k:", label=r"$h^2$"); ax[0].set(xlabel="grid spacing h (charge at height 1)", ylabel="|error of the reaction potential|", title="(a) second-order convergence"); ax[0].legend(frameon=False, fontsize=8)
    for e, hist in R["hist"].items(): ax[1].semilogy(hist, color={2.0: C["green"], 10.0: C["orange"], 80.0: C["red"]}[e], label=f"eps = {e:g}")
    ax[1].set(xlabel="conjugate-gradient iteration", ylabel="relative residual", title="(b) larger contrast, slower convergence"); ax[1].legend(frameon=False, fontsize=8)
    eps = [1.0] + [x[0] for x in R["more"][:2]]; allr = [x for x in rows if abs(x[2] - 1 / 4.5) < 1e-6] + [x for x in R["more"] if x[1] == 1.0]; es = [x[0] for x in allr]; its = [x[4] for x in allr]; o = np.argsort(es); ax[2].semilogx(np.array(es)[o], np.array(its)[o], "o-", color=C["pink"]); ax[2].set(xlabel="dielectric constant of the material", ylabel="CG iterations (16,250 unknowns)", title="(c) cost grows with the contrast"); plt.tight_layout(); return fig
def fig_energy(R, d=1.0):
    e = np.logspace(0, 3.2, 200); W = -(e - 1) / (e + 1) / (4 * d); fig, ax = plt.subplots(figsize=(6.4, 3.8)); ax.semilogx(e, W, color=C["blue"], lw=2, label="exact: $-\\frac{\\epsilon-1}{\\epsilon+1}\\frac{1}{4d}$")
    pts = [x for x in R["conv"] if abs(x[2] - d / 4.5) < 1e-6] + [x for x in R["more"] if x[1] == d]; ax.semilogx([p[0] for p in pts], [0.5 * p[5] for p in pts], "o", color=C["red"], label="grid + conjugate gradients"); ax.axhline(0, color=C["gray"], lw=2, ls="--", label="1/r pair interaction: 0")
    ax.axhline(-0.25, color="k", lw=0.8, ls=":"); ax.text(1.2, -0.235, "metal limit $-1/(4d)$", fontsize=8); ax.set(xlabel="dielectric constant", ylabel="energy of the charge near the surface [hartree]", title="a charge at height d = 1 above a dielectric"); ax.legend(frameon=False, fontsize=8, loc="center right"); plt.tight_layout(); return fig
def fig_slab(x, exact, phi_h, phi_a, xi):
    fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.6)); ax[0].axvspan(0, xi, color=C["blue"], alpha=0.12); ax[0].axvspan(xi, 1, color=C["orange"], alpha=0.18); ax[0].plot([0, *x, 1], [0, *exact, 1], "k-", lw=2, label="exact"); ax[0].plot(x, phi_h, "o", color=C["green"], label="harmonic mean"); ax[0].plot(x, phi_a, "s", color=C["red"], ms=4, label="arithmetic mean")
    ax[0].text(0.03, 0.8, "epsilon = 1", fontsize=8); ax[0].text(0.55, 0.5, "epsilon = 10", fontsize=8); ax[0].set(xlabel="position", ylabel="potential", title="(a) potential across two layers (no charge)"); ax[0].legend(frameon=False, fontsize=8, loc="lower right")
    ax[1].semilogy(x, np.maximum(np.abs(phi_h - exact), 1e-17), "o-", color=C["green"], label="harmonic mean"); ax[1].semilogy(x, np.abs(phi_a - exact), "s-", color=C["red"], ms=4, label="arithmetic mean"); ax[1].set(xlabel="position", ylabel="|error|", title="(b) only the harmonic mean is exact"); ax[1].legend(frameon=False, fontsize=8); plt.tight_layout(); return fig
def fig_sinebasis(T0, T1):
    fig, ax = plt.subplots(1, 2, figsize=(9.6, 4.0)); vm = 1.0
    for a, T, ttl in ((ax[0], T0, "(a) constant epsilon:\nthe sine waves are eigenvectors"), (ax[1], T1, "(b) dielectric half-space:\nthe stencil mixes the sine waves")):
        im = a.imshow(np.log10(np.abs(T) / np.abs(T).max() + 1e-6), cmap="viridis", vmin=-6, vmax=0); a.set(title=ttl, xlabel="sine wave", ylabel="sine wave", xticks=[], yticks=[])
    plt.colorbar(im, ax=ax, fraction=0.03, label="log10 of |matrix element| / largest"); return fig

In [ ]:
import types, time, sys, os
import numpy as np, matplotlib.pyplot as plt, scipy.sparse as sp, scipy.sparse.linalg as spla
np.set_printoptions(precision=5, suppress=True, linewidth=120)
D = types.SimpleNamespace(build=build, potential=potential, exact_potential=exact_potential); R = {}

## 1. What plain $1/r$ misses
A **dielectric** is an insulating material whose molecules can be polarized: in an electric field their charges shift slightly, creating a small opposite charge near the surface facing a nearby charge. That induced charge acts back on the original charge. The material is summarized by one number, the **dielectric constant** $\epsilon$ ($1$ for vacuum, about $2$ for oil, about $80$ for water). Place a unit charge at height $d$ above a flat surface with vacuum above and a dielectric below. The charge **attracts** its own induced charge, and the energy of that attraction is $W=-\frac{\epsilon-1}{\epsilon+1}\cdot\frac{1}{4d}$ (derived in Section 2). A sum of pair interactions $q_iq_j/r_{ij}$ over the charges you list has only *one* charge here, hence no pairs, and gives $W=0$. The interaction with the material is not a pair interaction between charges you listed; it is a response of the medium.
**Stop and predict:** for $\epsilon=4$ and $d=1$ bohr, is $W$ negligible or comparable to the repulsion $0.5$ hartree of two electrons $2$ bohr apart?

In [ ]:
d = 1.0
for eps2 in (1.0, 2.0, 4.0, 10.0, 80.0, 1e6):
    W = -(eps2 - 1) / (eps2 + 1) / (4 * d); print(f"epsilon = {eps2:9g}: interaction energy of the charge with the material W = {W:+.4f} hartree   (a sum of 1/r pair terms gives 0)")
print("for epsilon = 4 at 1 bohr the attraction is -0.15 hartree, a third of the repulsion of two electrons 2 bohr apart (0.5 hartree): not negligible; for a metal (epsilon -> infinity) it is -1/(4d)")
assert abs(-(4 - 1) / (4 + 1) / 4 + 0.15) < 1e-12

## 2. The exact answer: image charges
For a flat interface the induced charge can be replaced, for the potential in the vacuum, by a single **image charge** $q'=-\frac{\epsilon-1}{\epsilon+1}q$ at the mirror point (depth $d$ below the surface). Above the surface ($z>0$) the potential is that of the real charge plus the image, $\phi=\frac{q}{r_1}+\frac{q'}{r_2}$; below ($z<0$) it is that of a single charge $q''=\frac{2}{1+\epsilon}q$ at the real position, $\phi=\frac{q''}{r_1}$. These satisfy the two conditions that hold at any interface between media: the potential is **continuous**, and the **normal component of $\epsilon\nabla\phi$ is continuous** (the flux of the displacement field). The energy is $W=\tfrac12q\phi_R$ where $\phi_R=q'/(2d)$ is the **reaction potential**, the potential of the image at the charge (the factor $\tfrac12$ because the image is induced by the charge).
**Stop and predict:** the two conditions are both exact for this solution. How large will the differences between the two sides of the interface be, in a finite-difference check?

In [ ]:
eps2, d = 4.0, 1.0; qi, qt = image_charges(eps2); print(f"epsilon = {eps2:g}: image charge q' = {qi:+.4f}, transmitted charge q'' = {qt:+.4f}; reaction potential q'/(2d) = {qi / (2 * d):+.4f}; energy W = {0.5 * qi / (2 * d):+.4f}")
x0, y0 = np.array([0.3, 0.7, -0.5, 1.4]), np.array([0.2, -0.4, 0.9, 0.0]); dz = 1e-6; ph = lambda z: exact_potential(x0, y0, np.full_like(x0, z), eps2, d)
cont = np.abs(ph(+1e-9) - ph(-1e-9)).max(); flux = np.abs(((ph(dz) - ph(1e-9)) / (dz - 1e-9)) - eps2 * ((ph(-1e-9) - ph(-dz)) / (dz - 1e-9))).max()
print(f"continuity of the potential across the interface: largest jump {cont:.1e}\ncontinuity of the flux epsilon*dphi/dz: largest mismatch {flux:.1e} (finite-difference error)")
print("both conditions hold: the image solution is exact for the flat interface"); assert cont < 1e-6 and flux < 1e-4

## 3. The equation on a grid with a position-dependent $\epsilon$
In a medium the Poisson equation becomes $-\nabla\!\cdot(\epsilon\nabla\phi)=4\pi\rho$: the "conductance" $\epsilon$ multiplies the gradient. On the grid, the flux between a cell and its neighbour through the face between them is $\epsilon_f(\phi_i-\phi_j)/h$, and the equation at cell $i$ says that the net flux out equals the charge: $\sum_{\rm faces}\epsilon_f\frac{\phi_i-\phi_j}{h^2}=4\pi\rho_i$. The new question is which value $\epsilon_f$ to use on a face when the two cells have different $\epsilon$. Think of the two half-cells as two **resistors in series** (a larger $\epsilon$ is a better conductor of flux): the combination has the **harmonic mean** $\epsilon_f=\frac{2\epsilon_1\epsilon_2}{\epsilon_1+\epsilon_2}$. The arithmetic mean $(\epsilon_1+\epsilon_2)/2$ would over-count the better conductor.
**Stop and predict:** in a one-dimensional slab with $\epsilon=1$ on the left half and $\epsilon=10$ on the right half and $\phi=0,1$ at the two ends, which mean reproduces the exact piecewise-linear potential?

In [ ]:
def slab(N, eps_left, eps_right, mean):
    """1D chain of N interior nodes between phi = 0 (left) and phi = 1 (right); the interface lies between nodes; no charge"""
    h = 1.0 / (N + 1); e = np.where(np.arange(N + 2) < (N + 2) // 2, eps_left, eps_right).astype(float)
    f = 2 * e[:-1] * e[1:] / (e[:-1] + e[1:]) if mean == "harmonic" else 0.5 * (e[:-1] + e[1:])
    A = np.zeros((N, N)); b = np.zeros(N)
    for i in range(N):
        A[i, i] = f[i] + f[i + 1]
        if i > 0: A[i, i - 1] = -f[i]
        if i < N - 1: A[i, i + 1] = -f[i + 1]
    b[-1] += f[N] * 1.0; return np.linalg.solve(A, b), e, h
N = 9; x = np.arange(1, N + 1) / (N + 1); k = (N + 2) // 2
# exact: constant flux; potential drop across each half is proportional to 1/epsilon; interface at x = (k - 1/2) h
h = 1.0 / (N + 1); xi = (k - 0.5) * h; L1, L2 = xi, 1 - xi; e1, e2 = 1.0, 10.0; flux = 1.0 / (L1 / e1 + L2 / e2); exact = np.where(x <= xi, flux / e1 * x, flux / e1 * xi + flux / e2 * (x - xi))
for mean in ("harmonic", "arithmetic"):
    phi, _, _ = slab(N, e1, e2, mean); print(f"{mean:10s} mean: largest error against the exact piecewise-linear potential = {np.abs(phi - exact).max():.2e}")
ph_h, ph_a = slab(N, e1, e2, "harmonic")[0], slab(N, e1, e2, "arithmetic")[0]; fig_slab(x, exact, ph_h, ph_a, xi); plt.show()
print("the harmonic mean (resistors in series) reproduces the exact potential to rounding; the arithmetic mean does not"); assert np.abs(slab(N, e1, e2, "harmonic")[0] - exact).max() < 1e-12 and np.abs(slab(N, e1, e2, "arithmetic")[0] - exact).max() > 1e-3

## 4. Build the matrix and solve it by conjugate gradients
The grid is centred on the charge in $x,y$; the $z$ nodes sit half a cell off, so that the interface $z=0$ lies **midway between two node planes** (where the harmonic-mean face is exact). The charge is at height $d$ on a node. The unknowns are the potentials at the interior nodes; the values on the six walls of the box are taken from the exact solution and moved to the right-hand side (this tests the *interior* solver; the post comments on this choice). The matrix is symmetric and positive definite, so CG applies. We also solve the *same* problem in vacuum ($\epsilon=1$ everywhere): the difference of the two potentials at the charge is the reaction potential, in which the grid's own self-value of the charge cancels.
**Stop and predict:** how many iterations will CG need for a grid with a few thousand unknowns, if the iteration count grows in proportion to the number of cells along one side?

In [ ]:
eps2, d = 4.0, 1.0; h = d / 4.5; s = build(h, eps2, d); A = s["A"]
print(f"grid spacing {h:.4f}: {A.shape[0]} unknowns, {A.nnz / A.shape[0]:.2f} nonzeros per row; matrix symmetric: {abs(A - A.T).max() == 0}; smallest eigenvalue {spla.eigsh(A, k=1, which='SA', tol=1e-6)[0][0]:.4f} > 0 (positive definite)")
phi, it, hist = potential(s); print(f"conjugate gradients: {it} iterations; the residual falls from {hist[0]:.1f} to {hist[-1]:.1e}; every 40th value:", np.array(hist[::40]).round(8))
# the solution satisfies the discrete equations
res = np.linalg.norm(A @ phi[s["mask"].reshape(s["shape"])] - s["b"]) / np.linalg.norm(s["b"]); print(f"relative residual of the discrete equations, computed independently: {res:.1e}")
fig_potential(D); plt.show()
print("panel (a): the dielectric adds a negative potential, concentrated at the interface (the induced charge); panel (b): the potential along the axis follows the exact image solution (dots), and differs clearly from the vacuum potential (grey), which is what a 1/r pair interaction gives")
assert res < 1e-9 and it < 400

## 5. The reaction potential and its convergence
The exact reaction potential for $\epsilon=4$, $d=1$ is $-0.3000$. We compute it from the grid for three spacings and three dielectric constants. A method is of **order $p$** if its error falls like $h^p$ (a straight line of slope $p$ on a log-log plot).
**Stop and predict:** what slope will the error show?

In [ ]:
t0 = time.time(); d = 1.0; rows = []
for eps2 in (2.0, 4.0, 10.0):
    qi, _ = image_charges(eps2)
    for hh in (d / 2.5, d / 4.5, d / 8.5): phiR, it, n, _, _, _ = reaction(hh, eps2, d); rows.append((eps2, d, hh, n, it, phiR, qi / (2 * d)))
R["conv"] = rows
print("  eps     h      unknowns   CG its   reaction potential   exact      error"); [print(f"  {r[0]:4.1f}  {r[2]:.4f}  {r[3]:8d}  {r[4]:6d}     {r[5]:+.5f}        {r[6]:+.5f}   {r[5] - r[6]:+.2e}") for r in rows]
for e in (2.0, 4.0, 10.0):
    rr = [x for x in rows if x[0] == e]; er = np.abs([x[5] - x[6] for x in rr]); hs = np.array([x[2] for x in rr]); print(f"  eps = {e:g}: local slopes", np.round(np.diff(np.log(er)) / np.diff(np.log(hs)), 2))
print(f"[{time.time() - t0:.0f} s]  second order, as for the stencil of Post H1; the relative error at h = d/4.5 is about 0.3 % for every epsilon")
assert all(1.8 < s_ < 2.4 for e in (2.0, 4.0, 10.0) for s_ in np.diff(np.log(np.abs([x[5] - x[6] for x in rows if x[0] == e]))) / np.diff(np.log([x[2] for x in rows if x[0] == e])))

## 6. The dependence on $\epsilon$ and on the distance
Two more checks: the energy as the dielectric constant grows (towards a metal, $\epsilon\to\infty$, where $W\to-1/(4d)$), and the dependence on the distance $d$. Because the equations have no other length scale than $d$ and $h$, the *relative* error should depend only on the ratio $h/d$.
**Stop and predict:** keep $h/d=1/4.5$ and change $d$ from $0.5$ to $2$. Will the relative error change?

In [ ]:
t0 = time.time(); more = []
for eps2, dd in ((80.0, 1.0), (1000.0, 1.0), (4.0, 0.5), (4.0, 2.0)):
    hh = dd / 4.5; qi, _ = image_charges(eps2); phiR, it, n, _, _, _ = reaction(hh, eps2, dd); more.append((eps2, dd, hh, n, it, phiR, qi / (2 * dd)))
R["more"] = more; print("  eps     d     h       unknowns  CG its   reaction potential   exact      relative error"); [print(f"  {r[0]:6g} {r[1]:4.1f}  {r[2]:.4f}  {r[3]:8d}  {r[4]:5d}     {r[5]:+.5f}        {r[6]:+.5f}   {abs(r[5] / r[6] - 1):.4f}") for r in more]
R["hist"] = {}
for e in (2.0, 10.0, 80.0): s_ = build(1.0 / 4.5, e, 1.0); R["hist"][e] = cg(s_["A"], s_["b"], 1e-11)[2]
print(f"[{time.time() - t0:.0f} s]  the relative error is the same (0.3 %) at every distance when h/d is fixed, and the metal limit W -> -1/(4d) = -0.25 is approached: W(eps = 1000) = {0.5 * more[1][5]:+.4f}")
print("the number of CG iterations grows with the dielectric contrast (141 for eps = 2, 710 for eps = 80, 1574 for eps = 1000 on this grid): a strong contrast makes the matrix harder to invert iteratively (its condition number grows)")
fig_conv(R); plt.show(); fig_energy(R); plt.show()
assert max(abs(r[5] / r[6] - 1) for r in more[2:]) < 0.005 and abs(abs(more[2][5] / more[2][6] - 1) - abs(more[3][5] / more[3][6] - 1)) < 1e-3

## 7. Why the fast sine transform cannot be used here
For a uniform box with constant coefficients, the 7-point matrix is diagonalized by sine waves, which makes a *direct* three-step solve possible (Post H1). With a position-dependent $\epsilon$ that is no longer true: the stencil no longer multiplies a sine wave by a single number. We measure it on a small grid: express the matrix in the basis of products of sine waves and compute the fraction of the matrix norm that is **off the diagonal** (zero if the waves are eigenvectors).
**Stop and predict:** what fraction do you expect for constant $\epsilon$, and for a dielectric half-space?

In [ ]:
def stencil_eps(N, h, eps):
    idx = np.arange(N ** 3).reshape(N, N, N); A = np.zeros((N ** 3, N ** 3))
    for ax in range(3):
        for sgn in (1, -1):
            sa = [slice(None)] * 3; sb = [slice(None)] * 3
            if sgn == 1: sa[ax] = slice(0, -1); sb[ax] = slice(1, None)
            else: sa[ax] = slice(1, None); sb[ax] = slice(0, -1)
            a, b = tuple(sa), tuple(sb); c = (2 * eps[a] * eps[b] / (eps[a] + eps[b])) / h ** 2
            for i, j, cc in zip(idx[a].ravel(), idx[b].ravel(), c.ravel()): A[i, i] += cc; A[i, j] -= cc
        # (Dirichlet walls: the neighbours outside the box are not unknowns, but their faces still contribute to the diagonal)
    for ax in range(3):
        for end in (0, -1):
            sl = [slice(None)] * 3; sl[ax] = end; ids = idx[tuple(sl)].ravel(); ee = eps[tuple(sl)].ravel()
            for i, e_ in zip(ids, ee): A[i, i] += e_ / h ** 2
    return A
N = 5; h = 1.0; S1 = np.sqrt(2 / (N + 1)) * np.sin(np.pi * np.outer(np.arange(1, N + 1), np.arange(1, N + 1)) / (N + 1)); S3 = np.kron(np.kron(S1, S1), S1)
Ts = []
for name, eps in (("constant epsilon = 1", np.ones((N, N, N))), ("dielectric half-space (epsilon = 1 above, 10 below)", np.where(np.arange(N)[None, None, :] < 2, 10.0, 1.0) * np.ones((N, N, N)))):
    A5 = stencil_eps(N, h, eps); T = S3.T @ A5 @ S3; Ts.append(T); off = np.linalg.norm(T - np.diag(np.diag(T))) / np.linalg.norm(T); print(f"{name:55s}: fraction of the matrix norm off the diagonal in the sine basis = {off:.3f}")
fig_sinebasis(*Ts); plt.show()
print("constant epsilon: zero (the sine waves are eigenvectors, so the transform solves the system in three steps); with a dielectric: the waves are mixed by the stencil, the system is not diagonal in this basis, and the transform cannot solve it. Conjugate gradients, which needs only products with the matrix, does not care.")
assert off > 0.05

## 8. Your turn
**Exercise 1.** Use the arithmetic mean instead of the harmonic mean in `face_coefficient` and recompute the convergence table of Section 5. How do the error and the order change?
**Exercise 2.** Move the charge closer to the interface ($d=0.3$, with $h$ chosen so that $d/h$ is a half-integer). When does the grid stop resolving the image?
**Exercise 3.** Make the dielectric a *slab* of finite thickness (set $\epsilon=1$ again for $z<-t$). There is no simple image formula; what does the grid give for the reaction potential, and how does it approach the half-space value as $t$ grows?
**Exercise 4.** Replace the plain CG by CG preconditioned with the diagonal of $A$ (divide the residual by the diagonal). How does the iteration count change for $\epsilon=80$?
**Exercise 5.** Use open (monopole) boundary values instead of the exact potential on the walls and see how the result depends on the box size.

## 9. Self-check

In [ ]:
assert abs(-(4 - 1) / (4 + 1) / 4 + 0.15) < 1e-12 and res < 1e-9 and max(abs(r[5] / r[6] - 1) for r in more[2:]) < 0.005 and off > 0.05
print("All checks passed: you reproduced the numbers in the post.")

# Part II (optional): independent checks with SciPy
> **Read this first.** **(1)** The conjugate-gradient solution against a sparse **direct** solve (`scipy.sparse.linalg.spsolve`) of the same matrix. **(2)** The matrix is symmetric and positive definite (the conditions CG needs). **(3)** The limits of the exact result: $\epsilon\to1$ gives zero, $\epsilon\to\infty$ gives the metal value $-1/(2d)$ for the reaction potential. **(4)** Scale invariance: the relative error depends only on $h/d$. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
try:
    s_ = build(0.4, 4.0, 1.0); x_cg, it_, _ = cg(s_["A"], s_["b"], 1e-12); x_dir = spla.spsolve(s_["A"].tocsc(), s_["b"]); print(f"matrix with {s_['A'].shape[0]} unknowns: |CG - direct sparse solve| = {np.abs(x_cg - x_dir).max():.1e} (CG: {it_} iterations)")
    check("the conjugate-gradient solution equals a direct sparse solve (to 1e-8)", np.abs(x_cg - x_dir).max() < 1e-8)
    check("the matrix is exactly symmetric", abs(s_["A"] - s_["A"].T).max() == 0); lam_min = spla.eigsh(s_["A"], k=1, which="SA", tol=1e-8)[0][0]; check(f"the smallest eigenvalue ({lam_min:.4f}) is positive: positive definite", lam_min > 0)
except Exception as e_: print("direct-solve stage could not be completed:", type(e_).__name__, e_)
try:
    r1 = reaction(1.0 / 4.5, 1.0 + 1e-9, 1.0)[0]; rinf = reaction(1.0 / 4.5, 1e6, 1.0)[0]; print(f"epsilon -> 1: reaction potential {r1:+.2e} (exact 0); epsilon = 1e6: {rinf:+.4f} (metal: -0.5000)")
    check("epsilon -> 1 gives a vanishing reaction potential", abs(r1) < 1e-6); check("a huge dielectric constant approaches the metal value -1/(2d) = -0.5 (within 1.5 %)", abs(rinf + 0.5) < 0.0075)
    a_ = reaction(1.0 / 4.5, 4.0, 1.0)[0]; b_ = reaction(2.0 / 4.5, 4.0, 2.0)[0]; print(f"d = 1: {a_:+.5f} (exact -0.30000); d = 2: {b_:+.5f} (exact -0.15000): relative errors {abs(a_ / -0.3 - 1):.4f} and {abs(b_ / -0.15 - 1):.4f}")
    check("the relative error depends only on h/d (equal to 1e-3 for d = 1 and d = 2)", abs(abs(a_ / -0.3 - 1) - abs(b_ / -0.15 - 1)) < 1e-3)
except Exception as e_: print("limits stage could not be completed:", type(e_).__name__, e_)
if checks:
    ok_ = [l for l, p_ in checks if p_]; bad_ = [l for l, p_ in checks if not p_]; print(f"\nPart II: {len(ok_)} of {len(checks)} checks passed")
    for l in bad_: print("  not passed:", l)